# Q-format fixed-point, explored

VHDL doesn't have floats. Every signal is a fixed number of bits, and `Qm.f` is the usual notation for how those bits are interpreted: `m` integer bits (including the sign bit) and `f` fractional bits, so the value is `(bits read as an integer) / 2^f`.

This notebook pokes at `qformat.py` to build intuition for what rounding and saturation actually do to a value, before any of this shows up in VHDL.

## The scale factor

A register has no decimal point of its own — it just holds bits, which read as an ordinary integer. So a fractional value has to be faked: pick a fixed number of fractional bits `f`, multiply every real number by `2^f` before storing it, and remember to divide by `2^f` whenever you read it back. `2^f` is that multiplier, the "scale factor," and it's the same trick as writing money in cents instead of dollars because a till can only count whole units: $0.30 becomes 30 cents (scale factor 100), stored as the plain integer `30`.

Worked example at `frac_bits=4` (scale factor `2^4 = 16`, step size `1/16 = 0.0625`): `x = 0.3` scales to `0.3 * 16 = 4.8` — i.e. 0.3 sits at the 4.8-th tick on a ruler marked off in sixteenths. That's not an integer yet, so it gets rounded — `4.8` rounds to `5` — and `5` is the integer code actually stored. Reading it back: `5 / 16 = 0.3125`, not exactly `0.3`. That gap (`0.3125 - 0.3 = 0.0125`) is the quantization error, and it's inherent to forcing a real number onto a fixed grid — the rest of this notebook is about how that error behaves.

In [ ]:
import sys
sys.path.insert(0, '../models')

import numpy as np
import matplotlib.pyplot as plt
from qformat import to_int, from_int, saturate, quantize

## Quantization step

Storing a value as `Qm.f` snaps it to the nearest multiple of `2^-f`. Plotting `quantize(x)` against `x` makes the staircase visible.

In [ ]:
int_bits, frac_bits = 4, 2  # Q4.2: step size 0.25, range -8..7.75

x = np.linspace(-2, 2, 2000)
y = [quantize(v, int_bits, frac_bits) for v in x]

plt.plot(x, x, '--', label='ideal (float)', alpha=0.5)
plt.plot(x, y, label=f'quantized Q{int_bits}.{frac_bits}')
plt.xlabel('input')
plt.ylabel('output')
plt.legend()
plt.title('Quantization staircase')
plt.show()

## Rounding: nearest vs. truncate

`qformat.py` defaults to round-half-away-from-zero rather than truncation. Truncation here means floor (dropping the least-significant bits of a two's-complement number always rounds toward `-infinity`), so its error is one-sided: always `<= 0`, for every input, not just positive ones. Rounding's error is centered on zero instead.

In [ ]:
err_nearest = [quantize(v, int_bits, frac_bits, round_mode='nearest') - v for v in x]
err_truncate = [quantize(v, int_bits, frac_bits, round_mode='truncate') - v for v in x]

plt.plot(x, err_nearest, label='nearest')
plt.plot(x, err_truncate, label='truncate')
plt.axhline(0, color='gray', linewidth=0.5)
plt.xlabel('input')
plt.ylabel('quantization error')
plt.legend()
plt.title('Rounding mode changes the error distribution, not just its size')
plt.show()

print('mean error, nearest: ', np.mean(err_nearest))
print('mean error, truncate:', np.mean(err_truncate))

That's the whole reason the default matters: truncation's mean error isn't zero, so in a cascade of stages that bias compounds instead of averaging out.

## Saturation

Values outside the representable range get clamped, not wrapped — matching how the VHDL will saturate on overflow.

In [ ]:
x_wide = np.linspace(-12, 12, 2000)
y_sat = [quantize(v, int_bits, frac_bits) for v in x_wide]

plt.plot(x_wide, x_wide, '--', label='ideal (float)', alpha=0.5)
plt.plot(x_wide, y_sat, label=f'quantized + saturated Q{int_bits}.{frac_bits}')
plt.xlabel('input')
plt.ylabel('output')
plt.legend()
plt.title('Saturation flattens out past the representable range')
plt.show()

## The integer code underneath

`to_int`/`from_int` expose the raw integer that a register would actually hold — this is what later blocks use to generate bit-exact VHDL testbench vectors.

In [ ]:
for v in [0.0, 0.25, 1.0, -1.0, 3.9]:
    code = to_int(v, frac_bits)
    print(f'{v:>5} -> code {code:>4} -> back to float {from_int(code, frac_bits)}')